In [1]:
import wrds
import pandas as pd
from pathlib import Path

In [2]:
wrds_db = wrds.Connection()

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  too many connections for role "joshaijer"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
from pathlib import Path

base = Path.cwd()
data_dir = base / "DATA"
raw_dir = data_dir / "raw"
usa_dir = raw_dir / "USA"

usa_dir.mkdir(parents=True, exist_ok=True)

print("USA folder:", usa_dir)

raw_dir.mkdir(parents=True, exist_ok=True)


USA folder: /Users/joshaijer/Documents/factor_model_project/DATA/raw/USA


NameError: name 'processed_dir' is not defined

In [ ]:
desc = wrds_db.describe_table(library="contrib", table="global_factor")
available_cols = set(desc["name"].tolist())

print("Number of columns in contrib.global_factor:", len(available_cols))
desc[["name", "type"]].head(20)

Approximately 26624412 rows in contrib.global_factor.
Number of columns in contrib.global_factor: 443


,name,type
0,permno,DOUBLE PRECISION
1,permco,DOUBLE PRECISION
2,gvkey,VARCHAR(6)
3,iid,VARCHAR(3)
4,id,DOUBLE PRECISION
5,date,DATE
6,excntry,VARCHAR(3)
7,eom,DATE
8,source_crsp,DOUBLE PRECISION
9,size_grp,VARCHAR(5)


In [ ]:
paper_countries = [
    "USA","JPN","CHN","IND","KOR","HKG","TWN","FRA","GBR","THA",
    "AUS","SGP","SWE","ZAF","POL","ISR","VNM","ITA","TUR","CHE",
    "IDN","GRC","PHL","NOR","LKA","DNK","FIN","SAU","JOR","EGY",
    "ESP","KWT"
]

In [ ]:
id_cols = [
    "id",
    "eom",
    "excntry",
    "gvkey",
    "permno",
    "size_grp",
    "me"
]

In [ ]:
paper36_candidates = [
    "be_me",              # bm
    "at_gr1",             # agr
    "ret_1_0",            # mom_1
    "ret_6_1",            # mom_6
    "ret_12_1",           # mom_12
    "rmax1_21d",          # maxret
    "rvol_21d",           # retvol
    "ni_be",              # roe
    "ni_me",              # ep
    "sale_gr1",           # sgr
    "sale_me",            # sp
    "div12m_me",          # dy
    "ami_126d",           # ill
    "dolvol_126d",        # dolvol
    "dolvol_var_126d",    # stddolvol
    "turnover_126d",      # turn
    "turnover_var_126d",  # stdturn
    "be_gr1a",            # egr
    "debt_me",            # lev proxy
    "at_me",              # asset/market related predictor
    "resff3_6_1",         # residual momentum proxy
    "resff3_12_1",        # residual momentum proxy
    "niq_be",             # alternate roe-style variable
    "ocf_me",             # cfp-style proxy if available
    "fcf_me",             # cfp-style proxy if available
    "oaccruals_at",       # acc proxy if available
    "taccruals_at",       # acc / absacc proxy if available
    "oaccruals_ni",       # pctacc proxy
    "taccruals_ni",       # pctacc proxy
    "cop_at",             # cash profitability style proxy if available
    "ope_be",             # profitability style proxy if available
    "eqnpo_me",           # equity issuance / related
    "ebit_sale",          # profit margin style base
    "sale_be",            # sales/book type
    "debt_gr3",           # debt growth proxy
    "tax_gr1a"            # accounting growth style extra predictor
]


In [ ]:
feature_cols = [col for col in paper36_candidates if col in available_cols]
missing_cols = [col for col in paper36_candidates if col not in available_cols]

print("Requested candidate features:", len(paper36_candidates))
print("Available feature columns:", len(feature_cols))
print("Missing candidate columns:", len(missing_cols))

print("\nAvailable:")
print(feature_cols)

print("\nMissing:")
print(missing_cols)

Requested candidate features: 36
Available feature columns: 36
Missing candidate columns: 0

Available:
['be_me', 'at_gr1', 'ret_1_0', 'ret_6_1', 'ret_12_1', 'rmax1_21d', 'rvol_21d', 'ni_be', 'ni_me', 'sale_gr1', 'sale_me', 'div12m_me', 'ami_126d', 'dolvol_126d', 'dolvol_var_126d', 'turnover_126d', 'turnover_var_126d', 'be_gr1a', 'debt_me', 'at_me', 'resff3_6_1', 'resff3_12_1', 'niq_be', 'ocf_me', 'fcf_me', 'oaccruals_at', 'taccruals_at', 'oaccruals_ni', 'taccruals_ni', 'cop_at', 'ope_be', 'eqnpo_me', 'ebit_sale', 'sale_be', 'debt_gr3', 'tax_gr1a']

Missing:
[]


In [ ]:
all_cols = id_cols + feature_cols
select_clause = ", ".join(all_cols)

print("Total columns to download:", len(all_cols))
print(select_clause)

Total columns to download: 43
id, eom, excntry, gvkey, permno, size_grp, me, be_me, at_gr1, ret_1_0, ret_6_1, ret_12_1, rmax1_21d, rvol_21d, ni_be, ni_me, sale_gr1, sale_me, div12m_me, ami_126d, dolvol_126d, dolvol_var_126d, turnover_126d, turnover_var_126d, be_gr1a, debt_me, at_me, resff3_6_1, resff3_12_1, niq_be, ocf_me, fcf_me, oaccruals_at, taccruals_at, oaccruals_ni, taccruals_ni, cop_at, ope_be, eqnpo_me, ebit_sale, sale_be, debt_gr3, tax_gr1a


In [ ]:
start_year = 1963
end_year = 2023
n_years = end_year - start_year + 1

total_rows = 0
saved_files = []

for i, year in enumerate(range(start_year, end_year + 1), start=1):
    pct = 100 * i / n_years
    print(f"\n[{i}/{n_years}] {pct:.1f}% complete | Downloading USA {year}")
    
    sql_query = f"""
    SELECT {select_clause}
    FROM contrib.global_factor
    WHERE common = 1
      AND exch_main = 1
      AND primary_sec = 1
      AND obs_main = 1
      AND excntry = 'USA'
      AND eom >= '{year}-01-01'
      AND eom <= '{year}-12-31'
    """
    
    try:
        df_year = wrds_db.raw_sql(sql_query)
        n_rows = len(df_year)
        total_rows += n_rows
        
        print(f"Year rows: {n_rows:,}")
        print(f"Cumulative rows: {total_rows:,}")
        
        file_path = usa_dir / f"USA_{year}.parquet"
        df_year.to_parquet(file_path, index=False)
        saved_files.append(file_path.name)
        
        print(f"Saved: {file_path.name}")
        
    except Exception as e:
        print(f"Error in {year}: {e}")


[1/61] 1.6% complete | Downloading USA 1963


In [ ]:
saved = sorted(usa_dir.glob("USA_*.parquet"))
print(f"Number of saved yearly files: {len(saved)}")
print(saved[:5])
print(saved[-5:])

In [ ]:
usa_files = sorted(usa_dir.glob("USA_*.parquet"))

usa_dfs = []
for f in usa_files:
    print(f"Reading {f.name}")
    usa_dfs.append(pd.read_parquet(f))

usa_df = pd.concat(usa_dfs, ignore_index=True)

print("Final USA shape:", usa_df.shape)
usa_df.head()

In [ ]:
usa_df.to_parquet(usa_dir / "USA_full_sample.parquet", index=False)
print("Combined USA file saved.")